In [294]:
import warnings
warnings.filterwarnings('ignore')

In [295]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process

In [296]:
res = pd.read_csv("data/1976-2024-house.tab")
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
0,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,BILL DAVENPORT,DEMOCRAT,False,TOTAL,58906,157170,False,20250910,False
1,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,JACK EDWARDS,REPUBLICAN,False,TOTAL,98257,157170,False,20250910,False
2,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,WRITEIN,NaN,True,TOTAL,7,157170,False,20250910,False
3,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,J CAROLE KEAHEY,DEMOCRAT,False,TOTAL,66288,156362,False,20250910,False
4,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,"WILLIAM L \""BILL\"" DICKINSON",REPUBLICAN,False,TOTAL,90069,156362,False,20250910,False


In [297]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'runoff', 'special', 'candidate',
       'party', 'writein', 'mode', 'candidatevotes', 'totalvotes',
       'unofficial', 'version', 'fusion_ticket'], dtype=object)

In [298]:
res = res[(res['year'] >= 2018) & (res['stage'] == 'GEN') & (res['mode'] == 'TOTAL')]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False
28279,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,WRITEIN,NaN,True,TOTAL,163,242617,False,20250910,False
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False


In [299]:
res['party'].value_counts()

party
DEMOCRAT                   1711
REPUBLICAN                 1678
LIBERTARIAN                 389
INDEPENDENT                 184
CONSERVATIVE                 88
                           ... 
WRITE-IN (UNAFFILIATED)       1
POPULIST PARTY                1
ALOHA AINA PARTY              1
AMERICAN SHOPPING PARTY       1
REPUBLICAN, LIBERTARIAN       1
Name: count, Length: 131, dtype: int64

In [300]:
# Focus on two party vote share in the model - no need to wrangle with third parties for now
res = res[res['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False
28283,2018,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,NaN,False,MALLORY HAGAN,DEMOCRAT,False,TOTAL,83996,231915,False,20250910,False


In [301]:
res.shape

(3389, 20)

## FEC Data Wrangling

FEC data from: https://www.fec.gov/data/browse-data/?tab=bulk-data

In [302]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [303]:
webl18 = pd.read_table('data/fec/webl18.txt', sep='|', names=fec_webl_colnames)
webl20 = pd.read_table('data/fec/webl20.txt', sep='|', names=fec_webl_colnames)
webl22 = pd.read_table('data/fec/webl22.txt', sep='|', names=fec_webl_colnames)
webl24 = pd.read_table('data/fec/webl24.txt', sep='|', names=fec_webl_colnames)
webl24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,...,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0


In [304]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

cn18 = pd.read_table('data/fec/cn18.txt', sep='|', names=fec_cn_colnames)
cn20 = pd.read_table('data/fec/cn20.txt', sep='|', names=fec_cn_colnames)
cn22 = pd.read_table('data/fec/cn22.txt', sep='|', names=fec_cn_colnames)
cn24 = pd.read_table('data/fec/cn24.txt', sep='|', names=fec_cn_colnames)
cn24.head()

,CAND_ID,CAND_NAME,CAND_PTY_AFFILIATION,CAND_ELECTION_YR,CAND_OFFICE_ST,CAND_OFFICE,CAND_OFFICE_DISTRICT,CAND_ICI,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H0AK00105,"LAMB, THOMAS",NNE,2020,AK,H,0.0,C,N,C00607515,1861 W LAKE LUCILLE DR,NaN,WASILLA,AK,99654
1,H0AL01055,"CARL, JERRY LEE, JR",REP,2024,AL,H,1.0,I,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685
2,H0AL01097,"AVERHART, JAMES",DEM,2024,AL,H,2.0,C,C,C00708867,811 SPRINGHILL AV,NaN,MOBILE,AL,36602
3,H0AL02087,"ROBY, MARTHA",REP,2020,AL,H,2.0,I,P,C00462143,NaN,NaN,MONTGOMERY,NaN,NaN
4,H0AL02137,"DISMUKES, WILL",REP,2020,AL,H,2.0,O,P,C00714337,PO BOX 6811188,NaN,PRATTVILLE,AL,36068


In [305]:
def wrangle_fec(webl, cn, year):
    webl['cand_name_lst'] = webl['CAND_NAME'].str.split(',')
    cn['cand_name_lst'] = cn['CAND_NAME'].str.split(',')

    def refactor_str(cand_lst):
        if len(cand_lst) == 3:
            return cand_lst[1] + ' ' + cand_lst[0] + ' ' + cand_lst[2]
        elif len(cand_lst) == 1:
            return cand_lst[0]
        else:
            return cand_lst[1] + ' ' + cand_lst[0]

    webl['cand'] = webl['cand_name_lst'].map(refactor_str)
    cn['cand'] = cn['cand_name_lst'].map(refactor_str)

    df = pd.merge(left=webl, right=cn, on='CAND_ID', how='inner')
    df = df[[col for col in df.columns.values if ('_y' not in col)]]

    df.columns = df.columns.str.strip('_x')

    df['year'] = np.full(shape=(df.shape[0],), fill_value=year)

    return df

In [306]:
fec18 = wrangle_fec(webl18, cn18, 2018)
fec20 = wrangle_fec(webl20, cn20, 2020)
fec22 = wrangle_fec(webl22, cn22, 2022)
fec24 = wrangle_fec(webl24, cn24, 2024)
fec24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,...,2024,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501,2024
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,2024,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567,2024
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,...,2024,H,C,C00856716,PO BOX 242442,NaN,ANCHORAGE,AK,99524,2024
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,...,2024,H,C,C00866939,"2117 CHARINGWOOD DRIVE WEST, MOBIL",NaN,MOBILE,AL,366952916,2024
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,2024,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685,2024


In [307]:
fec = pd.concat([fec18, fec20, fec22, fec24], axis=0)
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H8AK00132,"SHEIN, DIMITRI",C,1,DEM,209916.04,0.0,209574.16,0.0,0.00,...,2018,H,C,C00646521,PO BOX 90787,NaN,ANCHORAGE,AK,99509.0,2018
1,H6AK00045,"YOUNG, DONALD E",I,2,REP,1234680.31,0.0,1387687.05,0.0,269726.86,...,2018,H,C,C00012229,2504 FAIR BANKS ST,NaN,ANCHORAGE,AK,99503.0,2018
2,H8AK01031,"NELSON, THOMAS JOHN",C,2,REP,9288.48,0.0,8821.97,0.0,0.00,...,2018,H,C,C00681155,PO BOX 670123,NaN,CHUGIAK,AK,99567.0,2018
3,H8AK00140,"GALVIN, ALYSE",C,3,IND,1949643.68,154.7,1943398.59,0.0,0.00,...,2018,H,C,C00665711,P.O. BOX 90020,NaN,ANCHORAGE,AK,99509.0,2018
4,H8AL01066,"KENNEDY, ROBERT JR.",C,1,DEM,166845.21,0.0,166845.21,0.0,0.00,...,2018,H,C,C00667949,312-T SCHILLINGER RD #116,NaN,MOBILE,AL,36608.0,2018


In [308]:
fec24.shape, fec.shape

((2373, 42), (10519, 42))

In [309]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'cand_name_lst', 'cand',
       'CAND_ELECTION_YR', 'CAND_OFFICE', 'CAND_STATUS', 'CAND_PCC',
       'CAND_ST1', 'CAND_ST2', 'CAND_CITY', 'CAND_ST', 'CAND_ZIP', 'year'],
      dtype=object)

In [310]:
fec[(fec['year'] == 2024) &
        (fec['CAND_OFFICE_ST'] == 'NY') &
        (fec['CAND_OFFICE_DISTRICT'] == 16) &
        (fec['CAND_PTY_AFFILIATION'] == 'DEM')]

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
1255,H0NY16143,"BOWMAN, JAMAAL",I,1,DEM,6052175.61,71801.12,5993807.77,0.0,21516.46,...,2024,H,C,C00709196,81 PONDFIELD ROAD,STE D 351,BRONXVILLE,NY,10708,2024
1256,H4NY16061,"GERALD, MICHAEL BERNARD DR.",C,1,DEM,41993.16,0.00,39994.98,0.0,0.00,...,2024,H,C,C00807123,15 WALLACE STREET,NaN,TUCKAHOE,NY,10707,2024
1257,H4NY16087,"LATIMER, GEORGE",C,1,DEM,6679191.82,0.00,6486537.83,0.0,0.00,...,2024,H,C,C00859041,47 WAINWRIGHT STREET,NaN,RYE,NY,10580,2024


In [311]:
process.extractOne('JOHN GRCAR', res['candidate'].values, scorer=fuzz.partial_ratio)[0]

'JOHN R CARTER'

In [312]:
fuzz.partial_ratio('JAMAAL BOWMAN', 'GEORGE LATIMER')

21.052631578947366

In [313]:
fuzz.token_sort_ratio('ERIC MICHAEL SWALWELL', 'ERIC SWALWELL')

76.47058823529412

In [314]:
def get_fuzzymatch_cand(year, state_po, district, party, candidate):
    '''
    :param party: Party name as noted in `res` dataframe.
    :param candidate: Candidate name as noted in the `res` dataframe.
    '''

    if party == 'DEMOCRAT':
        fec_party = 'DEM'
    elif party == 'REPUBLICAN':
        fec_party = 'REP'
    
    df = fec[(fec['year'] == year) &
        (fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district)]

    resdf = res[(res['year'] == year) &
        (res['state_po'] == state_po) &
        (res['district'] == district) &
        (res['candidate'] == candidate)]

    if party == 'DEM':
        cand_col_str = 'dem_cand'
    else:
        cand_col_str = 'rep_cand'

    # df['cand_fuzzymatch'] = df['CAND_NAME'].str.title().apply(
    #         lambda x: process.extractOne(x, mit_df[cand_col_str].values[0], scorer=fuzz.partial_ratio)[0]
    # )
    if resdf.shape[0] == 0:
        return ''

    if df.shape[0] == 0:
        #  raise ValueError('Candidate available in MIT Election Lab dataset, but not available in FEC dataset.')
        return 'Error'
    
    fuzzymatch = process.extractOne(resdf['candidate'].values[0], df['CAND_NAME'].values, scorer=fuzz.partial_ratio)
    return fuzzymatch[0]

In [315]:
get_fuzzymatch_cand(2018, 'AL', 1, 'DEMOCRAT', 'ROBERT KENNEDY JR')

'KENNEDY, ROBERT JR.'

In [316]:
get_fuzzymatch_cand(2024, 'CA', 14, 'DEMOCRAT', 'ERIC SWALWELL')

'SWALWELL, ERIC MICHAEL'

In [317]:
get_fuzzymatch_cand(2024, 'CA', 12, 'DEMOCRAT', 'JENNIFER TRAN')

'TRAN, JENNIFER'

In [318]:
get_fuzzymatch_cand(2024, 'MA', 4, 'REPUBLICAN', float('nan')) # running unopposed

''

In [319]:
get_fuzzymatch_cand(2020, 'CA', 28, 'REPUBLICAN', 'ERIC EARLY')

'EARLY, ERIC'

In [320]:
res['fec_candidate_name'] = res.apply(
    lambda x: get_fuzzymatch_cand(x['year'], x['state_po'], x['district'], x['party'], x['candidate']),
    axis=1
)
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,...,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket,fec_candidate_name
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,...,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False,"BYRNE, BRADLEY ROBERTS"
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,...,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False,"KENNEDY, ROBERT JR."
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,...,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False,"ROBY, MARTHA"
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,...,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False,"ISNER, TABITHA KAY"
28283,2018,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,NaN,...,MALLORY HAGAN,DEMOCRAT,False,TOTAL,83996,231915,False,20250910,False,"HAGAN, MALLORY"


In [321]:
res[res['fec_candidate_name'] == 'Error']

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,...,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket,fec_candidate_name


In [322]:
res.shape, fec.shape

((3389, 21), (10519, 42))

In [323]:
res = pd.merge(left=res, right=fec, left_on=['year', 'state_po', 'district', 'fec_candidate_name'], right_on=['year', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT', 'CAND_NAME'], how='left')
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,...,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,...,BRADLEY ROBERTS BYRNE,2018,H,C,C00545673,22489 SEA CLIFF DRIVE,NaN,FAIRHOPE,AL,36532.0
1,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,...,ROBERT JR. KENNEDY,2018,H,C,C00667949,312-T SCHILLINGER RD #116,NaN,MOBILE,AL,36608.0
2,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,...,MARTHA ROBY,2018,H,C,C00462143,3260 BANKHEAD AVE,NaN,MONTGOMERY,AL,361062448.0
3,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,...,TABITHA KAY ISNER,2018,H,C,C00657437,4349 WHITE ACRES RD.,NaN,MONTGOMERY,AL,36106.0
4,2018,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,NaN,...,MALLORY HAGAN,2018,H,C,C00668574,1810 WALNUT HILL LANE,NaN,OPELIKA,AL,36801.0


In [324]:
res.shape

(3400, 62)

In [326]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'runoff', 'special', 'candidate',
       'party', 'writein', 'mode', 'candidatevotes', 'totalvotes',
       'unofficial', 'version', 'fusion_ticket', 'fec_candidate_name',
       'CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'cand_name_lst', 'cand',
       'CAND_ELECTION_YR', 'CAND_OFFICE', 'CAND_STATUS', 'CAND_PCC',
       'CAND_ST1', 'CAND_ST2', 'CAND_CITY', 'CAND_ST', '

In [327]:
res[res.duplicated(subset=['year', 'state', 'special', 'district', 'candidate', 'TTL_INDIV_CONTRIB'])]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,...,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
2709,2024,CONNECTICUT,CT,9,16,1,US HOUSE,4,GEN,False,...,MICHAEL GOLDSTEIN,2024,H,C,C00858209,C/O RED CURVE SOLUTIONS,138 CONANT STREET SUITE 401,BEVERLY,MA,01915
2747,2024,FLORIDA,FL,12,59,43,US HOUSE,16,GEN,False,...,JAN SCHNEIDER,2024,H,C,C00447474,227 SEAGULL LN,NaN,SARASOTA,FL,34236
2870,2024,KANSAS,KS,20,47,32,US HOUSE,2,GEN,False,...,NANCY E BOYDA,2024,H,C,C00881177,1236 N 100 RD,NaN,BALDWIN CITY,KS,66006
2905,2024,LOUISIANA,LA,22,72,45,US HOUSE,6,GEN,False,...,ELBERT LEE GUILLORY,2024,H,C,C00882662,629 E. LANDRY ST.,NaN,OPELOUSAS,LA,70570
2917,2024,MARYLAND,MD,24,52,52,US HOUSE,2,GEN,False,...,KIMBERLY KLACIK,2024,H,C,C00726117,P.O. BOX 15361,200 WILSON POINT RD.,MIDDLE RIVER,MD,21220
2971,2024,MINNESOTA,MN,27,41,33,US HOUSE,3,GEN,False,...,TAD JUDE,2024,H,C,C00869974,PO BOX 1881,NaN,MAPLE GROVE,MN,55311
2975,2024,MINNESOTA,MN,27,41,33,US HOUSE,5,GEN,False,...,DALIA AL-AQIDI,2024,H,C,C00850636,8014 OLSON MEMORIAL HWY 55,#255,GOLDEN VALLEY,MN,55427
3177,2024,OREGON,OR,41,92,72,US HOUSE,3,GEN,False,...,JOANNA HARBOUR,2024,H,C,C00870451,27812 S HWY 211,NaN,ESTACADA,OR,97023
3219,2024,RHODE ISLAND,RI,44,15,5,US HOUSE,1,GEN,False,...,ALLEN WATERS,2024,H,C,C00830661,PO BOX 40565,NaN,PROVIDENCE,RI,02940
3375,2024,WASHINGTON,WA,53,91,73,US HOUSE,10,GEN,False,...,DON HEWETT,2024,H,N,C00794362,P.O. BOX 5833,NaN,LACEY,WA,98503


In [330]:
res[(res['state'] == 'MINNESOTA') & (res['year'] == 2024) & (res['district'] == 3)]['CAND_ICI']

2970    C
2971    O
Name: CAND_ICI, dtype: object

In [331]:
res[(res['state'] == 'OREGON') & (res['year'] == 2024) & (res['district'] == 3)]

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,...,cand,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
3176,2024,OREGON,OR,41,92,72,US HOUSE,3,GEN,False,...,JOANNA HARBOUR,2024,H,C,C00870451,P.O. BOX 1346,NaN,ESTACADA,OR,97023
3177,2024,OREGON,OR,41,92,72,US HOUSE,3,GEN,False,...,JOANNA HARBOUR,2024,H,C,C00870451,27812 S HWY 211,NaN,ESTACADA,OR,97023
3178,2024,OREGON,OR,41,92,72,US HOUSE,3,GEN,False,...,MAXINE DEXTER,2024,H,C,C00859108,PO BOX 12209,NaN,PORTLAND,OR,97212


In [332]:
res = res.drop_duplicates(subset=['state', 'year', 'district', 'candidate'], keep='last')
res.shape

(3389, 62)

In [333]:
res['inc'] = res['CAND_ICI'].map(lambda x: True if x == 'I' else False)
res['inc'].value_counts()

inc
False    1791
True     1598
Name: count, dtype: int64

## Pivoting Past Results

In [334]:
# aggfunc is sum to account for races where two or more candidates from the same party are running in the general,
# we sum the votes of these candidates together as part of calculating two-party vote share
data = pd.pivot_table(data=res, values='candidatevotes', columns=['party'], index=['year', 'state', 'state_po', 'special', 'district'], aggfunc='sum').reset_index()
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN
0,2018,ALABAMA,AL,False,1,89226.0,153228.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0
2,2018,ALABAMA,AL,False,3,83996.0,147770.0
3,2018,ALABAMA,AL,False,4,46492.0,184255.0
4,2018,ALABAMA,AL,False,5,101388.0,159063.0


In [335]:
data.duplicated(subset=['year', 'state', 'special', 'district']).any() # np.False_ --> no duplicates

np.False_

In [336]:
# Get candidates

def get_cands(year, state, special, district, party):
    res_sorted = res.sort_values(by=['year', 'state', 'district', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['district'] == district) &
        (res_sorted['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['candidate'].values[0]
    else:
        return repr(list(df['candidate'].values))

def get_incumbency_status(year, state, special, district, party):
    res_sorted = res.sort_values(by=['year', 'state', 'district', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['district'] == district) &
        (res_sorted['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['inc'].values[0]
    else:
        return [bool(x) for x in df['inc'].values]

def get_ttl_receipts(year, state, special, district, party): # Misnomer: It's actually grabbing total individual contributions, not total receipts
    res_sorted = res.sort_values(by=['year', 'state', 'district', 'candidate'], ascending=True)
    
    df = res_sorted[
        (res_sorted['year'] == year) &
        (res_sorted['state'] == state) &
        (res_sorted['special'] == special) &
        (res_sorted['district'] == district) &
        (res_sorted['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['TTL_INDIV_CONTRIB'].values[0]
    else:
        return [float(x) for x in df['TTL_INDIV_CONTRIB'].values]

def get_totvotes(year, state, special, district):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special) &
        (res['district'] == district)
    ]
    return df['totalvotes'].values[0]

In [337]:
data['totalvotes'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_totvotes(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,231915
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,230969
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,260673


In [338]:
data.shape

(1742, 8)

In [339]:
get_cands(2018, 'ALABAMA', False, 1, 'DEMOCRAT') # conventional

'ROBERT KENNEDY JR'

In [340]:
get_cands(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

"['JENNIFER TRAN', 'LATEEFAH SIMON']"

In [341]:
get_incumbency_status(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

[False, False]

In [342]:
get_ttl_receipts(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

[274389.14, 1913404.94]

In [343]:
get_cands(2024, 'Massachusetts'.upper(), False, 4, 'REPUBLICAN') # running unopposed

'[]'

In [344]:
get_cands(2020, 'California'.upper(), False, 28, 'REPUBLICAN')

'ERIC EARLY'

In [345]:
def get_dem_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'DEMOCRAT')

def get_rep_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'REPUBLICAN')

def get_dem_ici(year, state, special, district):
    return get_incumbency_status(year, state, special, district, 'DEMOCRAT')

def get_rep_ici(year, state, special, district):
    return get_incumbency_status(year, state, special, district, 'REPUBLICAN')

def get_dem_receipts(year, state, special, district):
    return get_ttl_receipts(year, state, special, district, 'DEMOCRAT')

def get_rep_receipts(year, state, special, district):
    return get_ttl_receipts(year, state, special, district, 'REPUBLICAN')

In [346]:
data['dem_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617,ROBERT KENNEDY JR,BRADLEY BYRNE
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230,TABITHA ISNER,MARTHA ROBY
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,231915,MALLORY HAGAN,MIKE ROGERS
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,230969,LEE AUMAN,ROBERT ADERHOLT
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,260673,PETER JOFFRION,MO BROOKS


In [347]:
data['dem_inc'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_ici(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_inc'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_ici(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617,ROBERT KENNEDY JR,BRADLEY BYRNE,False,True
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230,TABITHA ISNER,MARTHA ROBY,False,True
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,231915,MALLORY HAGAN,MIKE ROGERS,False,True
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,230969,LEE AUMAN,ROBERT ADERHOLT,False,True
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,260673,PETER JOFFRION,MO BROOKS,False,True


In [348]:
data['dem_funds'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_receipts(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_funds'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_receipts(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617,ROBERT KENNEDY JR,BRADLEY BYRNE,False,True,39095.21,343761.59
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230,TABITHA ISNER,MARTHA ROBY,False,True,501490.57,799289.56
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,231915,MALLORY HAGAN,MIKE ROGERS,False,True,409309.26,530372.0
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,230969,LEE AUMAN,ROBERT ADERHOLT,False,True,61160.87,468336.48
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,260673,PETER JOFFRION,MO BROOKS,False,True,553386.53,1216500.38


In [349]:
data = data.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
data.head()

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617,ROBERT KENNEDY JR,BRADLEY BYRNE,False,True,39095.21,343761.59
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230,TABITHA ISNER,MARTHA ROBY,False,True,501490.57,799289.56
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,231915,MALLORY HAGAN,MIKE ROGERS,False,True,409309.26,530372.0
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,230969,LEE AUMAN,ROBERT ADERHOLT,False,True,61160.87,468336.48
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,260673,PETER JOFFRION,MO BROOKS,False,True,553386.53,1216500.38


In [350]:
data['2party_votes'] = data['dem'] + data['rep']
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617,ROBERT KENNEDY JR,BRADLEY BYRNE,False,True,39095.21,343761.59,242454.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230,TABITHA ISNER,MARTHA ROBY,False,True,501490.57,799289.56,225810.0


In [351]:
data['dem_cand'] = data['dem_cand'].str.title()
data['rep_cand'] = data['rep_cand'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0


In [352]:
data['state'] = data['state'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes
0,2018,Alabama,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0
1,2018,Alabama,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0


In [353]:
# two-party vote share
data['dem_pct_2p'] = data['dem'] / data['2party_votes'] * 100
data['rep_pct_2p'] = data['rep'] / data['2party_votes'] * 100
data.head(2)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p
0,2018,Alabama,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0,36.801208,63.198792
1,2018,Alabama,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0,38.497409,61.502591


In [354]:
data['dem_tot_funds'] = data['dem_funds'].map(
    lambda x: x if isinstance(x, float) else (np.sum(np.array(x)) if (isinstance(x, list) and len(x) > 0) else 0)
)
data['rep_tot_funds'] = data['rep_funds'].map(
    lambda x: x if isinstance(x, float) else (np.sum(np.array(x)) if (isinstance(x, list) and len(x) > 0) else 0)
)
data.head()

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds
0,2018,Alabama,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59
1,2018,Alabama,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56
2,2018,Alabama,AL,False,3,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,False,True,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00
3,2018,Alabama,AL,False,4,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,False,True,61160.87,468336.48,230747.0,20.148474,79.851526,61160.87,468336.48
4,2018,Alabama,AL,False,5,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,False,True,553386.53,1216500.38,260451.0,38.927860,61.072140,553386.53,1216500.38


In [355]:
data['tot_funds'] = data['dem_tot_funds'] + data['rep_tot_funds']
data['dem_funds_2p_pct'] = data['dem_tot_funds'] / data['tot_funds'] * 100
data['rep_funds_2p_pct'] = data['rep_tot_funds'] / data['tot_funds'] * 100
data.head(3)

party,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,...,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2018,Alabama,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,...,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59,382856.80,10.211445,89.788555
1,2018,Alabama,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,...,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56,1300780.13,38.553062,61.446938
2,2018,Alabama,AL,False,3,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,...,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00,939681.26,43.558308,56.441692
